In [ ]:
%load_ext autoreload
%autoreload 2

# About this Notebook
This notebook is about testing the task loading and implementation as well as the PyTorch Dataset and DataLoader objects.

In [ ]:
import os
os.chdir("../..")
print(f"Set current working directory to '{os.getcwd()}' ...")

In [ ]:
from typing import List
from src.tasks.base_task import BaseTask
from src.tasks.denoising import DenoisingTask
from src.tasks.deraining import DerainingTask




In [ ]:

import torch
import logging
import matplotlib
import pandas as pd
import matplotlib.pyplot as plt

import src.utils.utils as utils
import src.tasks.taskloader as taskloader
import src.data.dataloader as dataloader
import src.train.training as training

from typing import List
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image, pil_to_tensor

from src.data.dataloader import get_metadata_df
from src.utils.utils import setup_logging
from src.dataset.general_dataset import NeuralizerGeneralDataset
from src.dataset.sampler import WeightedTaskSampler

# Set to logging.DEBUG for more information
setup_logging(loglevel=logging.INFO)
logger = logging.getLogger(__name__)

# Set logging level of external libraries to INFO to avoid cluttering
logging.getLogger('PIL').setLevel(logging.INFO)
logging.getLogger('matplotlib').setLevel(logging.INFO)


# Load CFG file
CFG: dict = utils.load_config("configs/config.yaml")

# Matplotlib settings
matplotlib.rcParams['mathtext.fontset'] = 'cm'  
matplotlib.rcParams['font.family'] = 'STIXGeneral'

# Load Metadata

In [ ]:
df = get_metadata_df(path="data/metadata.csv")
df.head(5)

# Test Base Tasks

In [ ]:
df.info()

In [ ]:
from src.tasks.base_task import BaseTask


from torchvision.transforms import v2

# Test Dataloader

In [ ]:
metadata_df = pd.read_csv(CFG["dataloader"]["metadata_location"])
metadata_df: pd.DataFrame = pd.read_csv(CFG["dataloader"]["metadata_location"])
tasks: List[BaseTask] = taskloader.get_tasks(df=metadata_df,  CFG=CFG["tasks"])

In [ ]:
train_dataloader, val_dataloader, test_dataloader = training.get_dataloaders(CFG=CFG, tasks=tasks)

In [ ]:
batch = next(iter(train_dataloader))
print(len(batch))  # How many elements returned?


In [ ]:
# Get a sample from the dataset
x, y, ctx_in, ctx_out, lossfun, class_names = next(iter(train_dataloader)) 

# Print information
print("x shape: ", x.shape)
print("y shape: ", y.shape)
print("ctx_in shape: ", ctx_in.shape)
print("ctx_out shape: ", ctx_out.shape)
print("loss function(s): ", lossfun)
print("class name: ", class_names)


# Visualize Input x and Output y for all the tasks

In [ ]:
import matplotlib.pyplot as plt
import torch

# --- 1. Collect One Sample Per Task ---
# We will iterate through the train_dataloader until we have 1 sample for every task
unique_tasks = {}
required_tasks = set([t.get_task_name() for t in tasks])

# --- Configuration ---
SAVE_PATH = "/path/to/save/figure.jpg"  
DPI = 300                             
IMG_SIZE_INCHES = 4                   

print(f"Searching for samples from: {required_tasks}...")

# Iterate through the dataloader
for batch in train_dataloader:
    # Unpack the batch 
    # images, labels, context_in, context_out, lossfuns, task_names = batch
    
    # Note: Dataloaders collate items, so 'task_names' is a tuple of strings
    images, labels, _, _, _, task_names = batch
    
    # Check each sample in the current batch
    for i, task_name in enumerate(task_names):
        if task_name not in unique_tasks:
            # Store the image and label (cpu, detach) for plotting
            unique_tasks[task_name] = (images[i].detach().cpu(), labels[i].detach().cpu())
            
    # Stop if we found all tasks
    if len(unique_tasks) == len(required_tasks):
        break

print(f"Found samples for: {list(unique_tasks.keys())}")

# --- 2. Setup Figure ---
num_tasks = len(unique_tasks)

# Calculate Figure Size
# Width = Num Tasks * Image Size
# Height = 2 Rows * Image Size + Extra space for text at bottom
fig, axes = plt.subplots(
    nrows=2, 
    ncols=num_tasks, 
    figsize=(num_tasks * IMG_SIZE_INCHES, 2.2 * IMG_SIZE_INCHES), # Slightly taller to fit text
    # wspace=0.05 -> Thin vertical gap
    # hspace=0.1 -> Gap between Input and Target rows
    gridspec_kw={'wspace': 0.05, 'hspace': 0.1}
)

# Handle single task edge case
if num_tasks == 1:
    axes = axes.reshape(2, 1)

def denormalize(tensor):
    # Adjust this if you use specific mean/std normalization
    return torch.clamp(tensor, 0, 1).permute(1, 2, 0).numpy()

# --- 3. Plotting Loop ---
for idx, (task_name, (img_tensor, lbl_tensor)) in enumerate(unique_tasks.items()):
    
    # --- Row 1: Input Image ---
    ax_img = axes[0, idx]
    ax_img.imshow(denormalize(img_tensor))
    ax_img.axis('off')

    # --- Row 2: Target Label ---
    ax_lbl = axes[1, idx]
    lbl_np = denormalize(lbl_tensor)
    
    if lbl_np.shape[-1] == 1:
        ax_lbl.imshow(lbl_np.squeeze(), cmap='gray')
    else:
        ax_lbl.imshow(lbl_np)
        

    ax_lbl.set_xticks([])
    ax_lbl.set_yticks([])
 
    for spine in ax_lbl.spines.values():
        spine.set_visible(False)
        
    # Add Task Name at the bottom of the column
    ax_lbl.set_xlabel(task_name, fontsize=14, fontweight='bold', labelpad=10)

# --- 4. Save and Show ---
os.makedirs(os.path.dirname(SAVE_PATH) if os.path.dirname(SAVE_PATH) else ".", exist_ok=True)

print(f"Saving high-res figure to {SAVE_PATH}...")
plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches='tight', pad_inches=0.1)
plt.show()

# Visualize Input x and Output y

In [ ]:
# Specify which batch to visualize
BATCH_IDX =2

def plot_input_output(x: torch.Tensor, y: torch.Tensor):
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(4, 5))
    axes[0].set_title("Input x")
    axes[0].imshow(to_pil_image(x))
    axes[1].set_title("Output y")
    axes[1].imshow(to_pil_image(y))
    fig.tight_layout()

if x.ndim > 3:
    plot_input_output(x[BATCH_IDX], y[BATCH_IDX])
else:
    plot_input_output(x, y)

# Visualize Context C

In [ ]:
from src.visualizations.contextvizualizer import plot_context_set, plot_context_set_overlay

plot_context_set(ctx_in[BATCH_IDX])
plot_context_set(ctx_out[BATCH_IDX])
# plot_context_set_overlay(ctx_in[BATCH_IDX], ctx_out[BATCH_IDX])

________
# Visualize ALL Train Tasks (seen during Training)

In [ ]:
from src.visualizations.contextvizualizer import plot_all_tasks

plot_all_tasks(CFG, tasks=tasks, context_set_size=8)

# Visualize ALL Test Tasks (not seen during Training)

In [ ]:
test_tasks: List[BaseTask] = taskloader.get_test_tasks(df=metadata_df,CFG=CFG["tasks"])
plot_all_tasks(CFG, tasks=test_tasks, context_set_size=8)